### Dosya yolları yükleme ve okuma

In [ ]:
import pandas as pd
veri = {}
dizin = "Your Path Here"
data_file = pd.read_csv(dizin , skiprows=0)
#y = data_file[data_file["Almanca"].duplicated(keep=False)]
res = data_file.to_dict(orient="list")
veri = dict(zip(res["Almanca"], res["Türkçe"]))
print(veri , len(veri))


{'Hallo': 'Merhaba', 'Tschüss': 'Hoşça kal', 'Bitte': 'Lütfen', 'Danke': 'Teşekkürler', 'Ja': 'Evet', 'Nein': 'Hayır', 'Vielleicht': 'Belki', 'Guten Morgen': 'Günaydın', 'Guten Abend': 'İyi akşamlar', 'Gute Nacht': 'İyi geceler', 'Entschuldigung': 'Afedersiniz', "Wie geht's?": 'Nasılsın', 'Gut': 'İyi', 'Schlecht': 'Kötü', 'Heute': 'Bugün', 'Morgen': 'Yarın', 'Gestern': 'Dün', 'Jetzt': 'Şimdi', 'Später': 'Daha sonra', 'Immer': 'Her zaman', 'Nie': 'Asla', 'Manchmal': 'Bazen', 'Hier': 'Burada', 'Dort': 'Orada', 'Alles': 'Her şey', 'Nichts': 'Hiçbir şey', 'Etwas': 'Bir şey', 'Jemand': 'Birisi', 'Niemand': 'Hiç kimse', 'Essen': 'Yemek', 'Trinken': 'İçmek', 'Wasser': 'Su', 'Brot': 'Ekmek', 'Kaffee': 'Kahve', 'Tee': 'Çay', 'Milch': 'Süt', 'Apfel': 'Elma', 'Haus': 'Ev', 'Zimmer': 'Oda', 'Tür': 'Kapı', 'Fenster': 'Pencere', 'Straße': 'Sokak', 'Auto': 'Araba', 'Bus': 'Otobüs', 'Bahn': 'Tren', 'Arbeit': 'İş', 'Schule': 'Okul', 'Lehrer': 'Öğretmen', 'Freund': 'Arkadaş', 'Familie': 'Aile', 'Mutter'

### OYUN BLOĞU

In [2]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
import html
state = {
    "kelimeler" : [] ,
    "mevcut_almanca" : None ,
    "mevcut_turkce" : None ,
    "skor" : {"dogru":{},"yanlis":{}}
}


### UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random

# ============================================================
#  STATE
# ============================================================
state = {
    "kelimeler": [],
    "mevcut_almanca": None,
    "mevcut_turkce": None,
    "skor": {"dogru": {}, "yanlis": {}},
}

# ============================================================
#  WIDGETS
# ============================================================
soru_label = widgets.HTML(value="<h2 style='text-align:center'>Yükleniyor...</h2>")
skor_label = widgets.HTML(value="")
mesaj_label = widgets.HTML(value="")

sik_butonlari = [
    widgets.Button(description="...", layout=widgets.Layout(width="250px", margin="3px"))
    for _ in range(4)
]

sonraki_buton = widgets.Button(
    description="Sonraki →",
    button_style="info",
    layout=widgets.Layout(width="150px", margin="10px", display="none")
)
rapor_buton = widgets.Button(
    description="📊 Raporu Göster",
    button_style="warning",
    layout=widgets.Layout(width="200px", margin="10px", display="none")
)
rapor_cikti = widgets.Output()


# ============================================================
#  FONKSIYONLAR
# ============================================================
def skor_label_guncelle():
    d = len(state["skor"]["dogru"])
    y = len(state["skor"]["yanlis"])
    skor_label.value = f"<p style='text-align:center'>✅ Doğru: {d}  |  ❌ Yanlış: {y}</p>"


def sonraki_kart():
    if not state["kelimeler"]:
        oyun_bitti()
        return
    
    # Yeni kart
    almanca = state["kelimeler"].pop()
    turkce = veri[almanca]
    state["mevcut_almanca"] = almanca
    state["mevcut_turkce"] = turkce
    
    # 4 şık üret
    diger_cevaplar = [v for v in veri.values() if v != turkce]
    yanlis_3 = random.sample(diger_cevaplar, 3)
    secenekler = [turkce] + yanlis_3
    random.shuffle(secenekler)
    
    # UI güncelle
    soru_label.value = f"<h2 style='text-align:center'>{almanca}</h2>"
    for buton, sik in zip(sik_butonlari, secenekler):
        buton.description = sik
        buton.disabled = False
        buton.button_style = ""
    
    mesaj_label.value = ""
    sonraki_buton.layout.display = "none"
    skor_label_guncelle()


def cevap_kontrol(b):
    secilen = b.description
    dogru = state["mevcut_turkce"]
    almanca = state["mevcut_almanca"]
    
    if secilen == dogru:
        b.button_style = "success"
        state["skor"]["dogru"][almanca] = dogru
        mesaj_label.value = "<p style='text-align:center; color:green'>✓ Doğru!</p>"
    else:
        b.button_style = "danger"
        # Doğru olanı yeşil yap
        for buton in sik_butonlari:
            if buton.description == dogru:
                buton.button_style = "success"
        state["skor"]["yanlis"][almanca] = dogru
        mesaj_label.value = f"<p style='text-align:center; color:red'>✗ Yanlış. Doğrusu: <b>{dogru}</b></p>"
    
    # Tüm butonları kilitle
    for buton in sik_butonlari:
        buton.disabled = True
    
    # Sonraki butonu göster
    sonraki_buton.layout.display = ""


def sonraki_tiklandi(b):
    sonraki_kart()


def oyun_bitti():
    soru_label.value = "<h2 style='text-align:center'>🎉 Oyun Bitti!</h2>"
    for buton in sik_butonlari:
        buton.layout.display = "none"
    mesaj_label.value = ""
    sonraki_buton.layout.display = "none"
    rapor_buton.layout.display = ""


def raporu_goster(b):
    import pandas as pd
    
    with rapor_cikti:
        clear_output()
        d = len(state["skor"]["dogru"])
        y = len(state["skor"]["yanlis"])
        toplam = d + y
        yuzde = (d / toplam) * 100 if toplam > 0 else 0
        
        print("=" * 60)
        print(f"   RAPOR — Doğruluk: %{yuzde:.1f} ({d}/{toplam})")
        print("=" * 60)
        
        if state["skor"]["yanlis"]:
            en_uzun = max(len(k) for k in state["skor"]["yanlis"].keys())
            print("\nYanlış bildiğin kelimeler:")
            for almanca, turkce in state["skor"]["yanlis"].items():
                print(f"  - {almanca:<{en_uzun}} → {turkce}")
        else:
            print("\n🎉 Tebrikler, hiç yanlış yapmadın!")
        
        # CSV kayıt
        dogru_df = pd.DataFrame(state["skor"]["dogru"].items(), columns=["Almanca", "Türkçe"])
        dogru_df.to_csv("dogrular.csv", index=False, encoding="utf-8")
        
        yanlis_df = pd.DataFrame(state["skor"]["yanlis"].items(), columns=["Almanca", "Türkçe"])
        yanlis_df.to_csv("yanlislar.csv", index=False, encoding="utf-8")
        
        print("\n✅ Dosyalar kaydedildi: dogrular.csv & yanlislar.csv")


def oyunu_baslat():
    state["kelimeler"] = list(veri.keys())
    random.shuffle(state["kelimeler"])
    state["skor"] = {"dogru": {}, "yanlis": {}}
    
    rapor_buton.layout.display = "none"
    for buton in sik_butonlari:
        buton.layout.display = ""
    
    sonraki_kart()


# ============================================================
#  EVENT BAĞLAMA
# ============================================================
for buton in sik_butonlari:
    buton.on_click(cevap_kontrol)

sonraki_buton.on_click(sonraki_tiklandi)
rapor_buton.on_click(raporu_goster)


# ============================================================
#  YERLEŞİM VE BAŞLAT
# ============================================================
ana_konteyner = widgets.VBox([
    skor_label,
    soru_label,
    widgets.VBox(sik_butonlari),
    mesaj_label,
    sonraki_buton,
    rapor_buton,
    rapor_cikti,
])

display(ana_konteyner)
oyunu_baslat()